In [1]:
from utils_phoneme_reco import *

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


# Wav2vec+CTC

In [9]:

MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cuda"
model = model.to(device)
model.eval()


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [8]:
!wc -l /vol/corpora/Rhapsodie/TextGrid.phones.sampa

0 /vol/corpora/Rhapsodie/TextGrid.phones.sampa
wc: /vol/corpora/Rhapsodie/TextGrid.phones.sampa: Is a directory


# WavLM+CTC

In [2]:
import torch, librosa
from transformers import WavLMForCTC, Wav2Vec2FeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

#CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme/checkpoint-268600"
CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large/checkpoint-476220"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large"          # where vocab.json / tokenizer were saved

device = "cuda"
model = WavLMForCTC.from_pretrained(CKPT).to(device).eval()
feat  = Wav2Vec2FeatureExtractor.from_pretrained(CKPT)   # also present in CKPT
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"
model = model.to(device)
model.eval()

WavLMForCTC(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMLayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
 

# Whisper+CTC

In [2]:
#whisper
from transformers import WhisperFeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme/checkpoint-193924"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme"
device = "cuda"
model = WhisperEncoderForCTC.from_pretrained(CKPT).to(device).eval()
feat  = WhisperFeatureExtractor.from_pretrained(CKPT)
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'WhisperTokenizer'. 
The class this function is called from is 'Wav2Vec2PhonemeCTCTokenizer'.


In [3]:
import os
import json
from pathlib import Path
import pandas as pd
import pickle

style = pd.read_csv("/vol/corpora/Rhapsodie/wav_style.csv")
audio_dir = "/vol/corpora/Rhapsodie/wav16k_corrected_withsilence"
textgrid_dir = Path("/vol/corpora/Rhapsodie/TextGrid.phones.sampa")
model_name = "whisper"

ref_inventory = set()
hyp_inventory = set()
alignment_store = {}

for filepath in textgrid_dir.glob("*"):
    print(filepath)
    f = filepath.stem[:-4] + ".wav"
    audio_path = os.path.join(audio_dir, f)

    if os.path.exists(audio_path) and f !="Rhap-D2004.wav":
        s = list(
            style[style["file"] == f.split("-")[1].split(".")[0]]["style"]
        )[0]

        textgrid_path = os.path.join(
            textgrid_dir,
            f.replace(".wav", "-Pro.TextGrid")
        )
        if model_name =="whisper":
            pred_phonemes, pred_alignments = get_phoneme_alignments_whisper_ctcfa(model, feat,tok, audio_path)
        elif model_name == "wavlm":
            pred_phonemes, pred_alignments = get_phoneme_alignments_wavlm_ctcfa(model, feat,tok, audio_path)
        else:
            pred_phonemes, pred_alignments = get_phoneme_alignments_w2v_ctcfa(model, processor, audio_path)

        ref_alignments = get_reference_alignments(
            textgrid_path,
            t="sampa"
        )

        clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
        clean_ref = clean_alignment_dict(
            ref_alignments,
            flag="",
            is_hyp=False
        )

        ref_seq = extract_phoneme_sequence(clean_ref)
        hyp_seq = extract_phoneme_sequence(clean_hyp)
        # collect inventories
        for ph in ref_seq:
            ref_inventory.add(ph)

        for ph in hyp_seq:
            hyp_inventory.add(ph)

        entry = {
            "file": f,
            "style": s,
            "ref_intervals": clean_ref,#get_ref_intervals_rhap(clean_ref, audio_path),
            "hyp_intervals": clean_hyp, #get_hyp_intervals(clean_hyp),
            "ref_seq": ref_seq,
            "hyp_seq": hyp_seq,
        }

        alignment_store[f] = entry



Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/Rhapsodie/TextGrid.phones.sampa/Rhap-D0020-Pro.TextGrid


/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/utils_phoneme_reco.py:120: UserWarning: torchaudio.functional._alignment.forced_align has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  aligned, _ = torchaudio.functional.forced_align(
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.

/vol/corpora/Rhapsodie/TextGrid.phones.sampa/Rhap-D2002-Pro.TextGrid


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


...

/vol/corpora/Rhapsodie/TextGrid.phones.sampa/Rhap-M0012-Pro.TextGrid


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/Rhapsodie/TextGrid.phones.sampa/Rhap-M0008-Pro.TextGrid


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/Rhapsodie/TextGrid.phones.sampa/Rhap-M0010-Pro.TextGrid


In [4]:
# ==========================
# Inventory comparison
# ==========================

print(f"Reference inventory size: {len(ref_inventory)}")
print(f"Hypothesis inventory size: {len(hyp_inventory)}")

only_in_ref = sorted(ref_inventory - hyp_inventory)
only_in_hyp = sorted(hyp_inventory - ref_inventory)

if not only_in_ref and not only_in_hyp:
    print("\n✓ REF and HYP inventories are identical.")
else:
    print("\n✗ Inventories differ.")

    if only_in_ref:
        print("\nPhonemes present only in REF:")
        print(only_in_ref)

    if only_in_hyp:
        print("\nPhonemes present only in HYP:")
        print(only_in_hyp)

# Optional: print full inventories
print("\nREF inventory:")
print(sorted(ref_inventory))

print("\nHYP inventory:")
print(sorted(hyp_inventory))

Reference inventory size: 34
Hypothesis inventory size: 34

✗ Inventories differ.

Phonemes present only in REF:
['ɥ']

Phonemes present only in HYP:
['ŋ']

REF inventory:
['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɥ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']

HYP inventory:
['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'ŋ', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']


In [5]:
from metrics import *
import pickle
from jiwer import process_words
# (optional) save back so you don't redo it
with open(f"ctc_results/alignment_{model_name}_rhap.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [6]:
from metrics_alignment import *

with open(f"ctc_results/alignment_{model_name}_rhap.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"ctc_results/metrics++_{model_name}_rhap.csv", per_phoneme_csv=None)


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,Rhap-D0020.wav,spont,251,62.347978,40.8625,42.150568,42.9175,82.545388,79.8625,36.454183,...,20.000,16.334661,297,275,17.171717,27,5,0.0,48.601399,82.867133
1,Rhap-D2002.wav,planned,1863,49.904442,33.0625,40.568179,35.3125,59.240705,77.7325,26.355341,...,20.000,16.800859,2519,2228,28.066693,342,51,0.0,41.668422,78.491679
2,Rhap-M0009.wav,spont,471,57.859952,44.8075,47.265812,47.6875,68.454092,86.8125,43.949045,...,23.300,18.683652,555,519,16.036036,41,5,0.0,36.871508,76.350093
3,Rhap-D0017.wav,spont,288,60.440252,47.4000,49.283576,48.7000,71.596927,87.7500,47.916667,...,25.950,18.750000,319,310,11.598746,15,6,0.0,40.381558,71.542130
4,Rhap-D2006.wav,planned,810,58.135225,35.8400,38.616438,38.3875,77.654012,85.0625,30.802469,...,30.000,23.456790,943,888,15.164369,65,10,0.0,44.565811,78.427089
5,Rhap-M0002.wav,spont,413,50.499831,31.6625,36.776071,34.7375,64.223590,84.7375,26.634383,...,22.100,21.307506,529,471,22.495274,61,3,0.0,42.200000,80.000000
6,Rhap-D2009.wav,planned,2759,48.710219,37.8125,42.995617,38.8875,54.424821,78.0125,32.149329,...,20.000,17.796303,3062,2967,11.724363,151,56,0.0,33.239343,78.387792
7,Rhap-M0023.wav,spont,549,62.347108,28.6875,36.897172,32.0375,87.797045,82.6125,22.768670,...,20.000,19.489982,735,652,27.755102,101,18,0.0,44.844989,82.624369
8,Rhap-D0005.wav,spont,1705,54.415340,27.6875,40.404295,29.0625,68.426386,70.2975,20.586510,...,20.000,17.712610,2412,2029,31.094527,426,43,0.0,44.764693,78.766044
9,Rhap-M0016.wav,spont,514,68.587865,48.1625,53.763278,50.0250,83.412451,100.2375,48.540856,...,30.000,26.070039,633,585,21.011058,62,14,0.0,37.931034,73.070608


In [7]:
dict_to_csv(alignment_store, f'ctc_results/pred_{model_name}_rhap.csv')

# Prepare for MFA

In [8]:
df1=pd.read_csv(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/ctc_results/pred_{model_name}_rhap.csv")
df1["filename1"]=[i.split("-")[1].split(".")[0]+"-"+i.split("-")[0]+".wav" for i in df1["filename"]]
df1["speaker_id"]=[i.split("-")[0] for i in df1["filename1"]]
df1 = df1[df1["speaker_id"] != "D2004"]

In [9]:
df1["predicted_phonemes"] = df1["predicted_phonemes"].apply(normalize_phones)
ref_inventory = set()

for seq in df1["predicted_phonemes"].dropna():
    tokens = seq.split()
    ref_inventory.update(tokens)

print(sorted(ref_inventory))
print("Number of unique phonemes:", len(ref_inventory))

['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'ŋ', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Number of unique phonemes: 34


In [10]:
#Generation des fichiers pour mFA
import os
import torch
import torchaudio
import shutil

corpus_dir = f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/mfa_rhap_{model_name}_ctc"
if os.path.exists(corpus_dir):
    shutil.rmtree(corpus_dir)  
os.makedirs(corpus_dir)
target_sr = 16000
wav_path = "/vol/corpora/Rhapsodie/wav16k_corrected_withsilence"
for idx, row in df1.iterrows():
    audio_path = os.path.join(wav_path, row["filename"])
    tmp = os.path.join(wav_path, row["filename1"])
    tokens = row["predicted_phonemes"]
    utt_id = os.path.splitext(os.path.basename(tmp))[0]
    speaker_id = str(row["speaker_id"])  

    # Create speaker folder
    speaker_dir = os.path.join(corpus_dir, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # Save wav inside speaker folder
    torchaudio.save(
        os.path.join(speaker_dir, f"{utt_id}.wav"),
        waveform,
        target_sr
    )

    # Save lab inside speaker folder
    with open(os.path.join(speaker_dir, f"{utt_id}.lab"), "w", encoding="utf-8") as f:
        f.write(tokens.strip())

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch

In [11]:
with open(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/output_ph_reco/phoneme_rhap_{model_name}_ctc.txt", "w", encoding="utf-8") as f:
    for ph in ref_inventory:
        f.write(f"{ph} {ph}\n")

# MFA ALIGN: GO to cmd 

# MFA alignment

In [12]:
#rhapsodie

import os
import json
from pathlib import Path
import pandas as pd
import pickle
from praatio import textgrid
import os
import json
from pathlib import Path
import pandas as pd
import pickle
style = pd.read_csv("/vol/corpora/Rhapsodie/wav_style.csv")
tg_path = Path(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/align_{model_name}_rhap")
audio_dir = "/vol/corpora/Rhapsodie/wav16k_corrected_withsilence"
textgrid_dir = Path("/vol/corpora/Rhapsodie/TextGrid.phones.sampa")
ref_files = textgrid_dir.glob("*")
full_paths = list(tg_path.rglob("*.TextGrid"))
ref_dict={str(f.stem)[:-4].split("-")[1]+"-"+str(f.stem)[:-4].split("-")[0]:str(f) for f in ref_files}
alignment_store = {}
for hyp_path in full_paths:
    if ".ipynb_checkpoints" not in str(hyp_path):
        stem=hyp_path.stem
        #stem = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0] #txtgrid from mfa
        if stem not in ref_dict.keys() or stem=="D2004-Rhap":
            continue
    
        ref_path = ref_dict[stem]
        
        #f = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0] + ".wav"
        f = stem.split("-")[1]+"-"+stem.split("-")[0] + ".wav"
        audio_path = os.path.join(audio_dir, f)
        if os.path.exists(audio_path):
            s = list(style[style["file"] == f.split("-")[1].split(".")[0]]["style"])[0]
            phones_hyp, hyp_intervals = extract_phones_from_textgrid(hyp_path, t="phones")
            ref_alignments = get_reference_alignments(
            ref_path,
            t="sampa"
        )
            pred_alignments=[]
            for i,j,k in hyp_intervals:
                pred_alignments.append({"phoneme":k,"start":i,"end":j})
            
            clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
            clean_ref = clean_alignment_dict(ref_alignments,  is_hyp=False)
            
            
            entry = {
                "file":          f,
                "style":s,
                "ref_intervals": clean_ref,
                "hyp_intervals": clean_hyp,
                "ref_seq":       extract_phoneme_sequence(clean_ref),
                "hyp_seq":       extract_phoneme_sequence(clean_hyp),
            }
            alignment_store[f] = entry 

In [13]:
from metrics import *
import pickle
from jiwer import process_words
# (optional) save back so you don't redo it
with open(f"mfa_results/alignment_{model_name}_mfa_rhap.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [14]:
from metrics_alignment import *

with open(f"mfa_results/alignment_{model_name}_mfa_rhap.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"mfa_results/metrics++_{model_name}_mfa_rhap.csv", per_phoneme_csv=None)


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,Rhap-D0003.wav,spont,2015,40.560113,11.9980,39.520237,12.0030,41.599989,67.0518,15.434243,...,21.0010,23.027295,2605,2398,25.412668,279,72,0.239856,65.400760,83.110134
1,Rhap-M0023.wav,spont,549,37.163143,10.4980,37.558179,10.5020,36.768107,53.4427,10.837887,...,13.8940,13.114754,735,652,27.755102,101,18,1.009373,74.116799,87.382841
2,Rhap-D2013.wav,spont,2116,12.029679,7.3015,11.720216,7.3975,12.339142,25.0010,2.079395,...,10.0030,4.631380,2227,2227,6.151774,26,26,1.436911,84.687921,97.215986
3,Rhap-D2010.wav,spont,2066,46.066322,9.7000,45.414342,10.0040,46.718302,60.9979,12.173282,...,19.9790,15.537270,2837,2407,29.044766,483,53,0.038139,73.302822,84.820748
4,Rhap-D0008.wav,spont,1432,39.554594,12.2510,39.553513,12.0065,39.555675,49.9970,9.916201,...,20.0030,14.385475,1736,1560,18.951613,201,25,0.060680,70.509709,88.956311
5,Rhap-D2003.wav,spont,2908,93.034257,10.9965,92.593097,10.9955,93.475416,71.9900,14.047455,...,19.9960,16.540578,3943,3461,28.328684,564,82,0.135062,69.259860,82.549973
6,Rhap-M0022.wav,spont,537,108.641921,13.9000,110.278372,13.9990,107.005469,64.5996,14.152700,...,20.0060,20.297952,652,591,18.711656,68,7,0.000000,66.291231,84.794851
7,Rhap-M0002.wav,spont,413,83.042282,13.2000,83.299569,13.7010,82.784995,97.7700,18.765133,...,21.0010,23.728814,529,471,22.495274,61,3,0.000000,65.000000,81.600000
8,Rhap-D2007.wav,spont,1788,90.808847,11.9960,92.451100,11.8980,89.166593,59.9985,11.633110,...,19.9970,15.212528,2533,2177,32.806948,442,86,0.254777,69.214437,83.906582
9,Rhap-D2011.wav,planned,2878,22.645459,10.0005,21.728561,10.9905,23.562356,44.0045,8.234885,...,19.9960,13.829048,3324,3111,14.380265,245,32,0.062160,74.498834,90.349650
